# Streamlined CRISPR Screen Analysis Tutorial

This tutorial walks through the Scanpy-style workflow provided by ``streamlined_crispr``.

## Prerequisites

Install the project in editable mode (with the optional ``test`` extras) to make the package importable and provide AnnData, NumPy, pandas, and SciPy:

```bash
pip install -e .[test]
```

## Prepare a demo dataset

The repository includes a small synthetic perturbation screen stored at ``data/demo_benchmark.h5ad``. Run ``python benchmarking/generate_demo_dataset.py`` to regenerate it if necessary.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))
DATA_PATH = ROOT / 'data' / 'demo_benchmark.h5ad'
OUTPUT_DIR = ROOT / 'notebooks' / 'tutorial_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import streamlined_crispr as scr
from benchmarking.generate_demo_dataset import write_demo_dataset

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    write_demo_dataset(DATA_PATH)

## Inspect the on-disk AnnData

``scr.read_h5ad_ondisk`` returns a read-only :class:`scr.AnnData` wrapper. The ``obs`` and ``var`` accessors expose ``head`` and ``load`` helpers so you can preview metadata without materialising the entire object.

In [2]:
adata_ro = scr.read_h5ad_ondisk(DATA_PATH)
adata_ro

AnnData object with n_obs × n_vars = 400 × 100 backed at '/Users/dujinhong/Library/CloudStorage/OneDrive-TheUniversityOfHongKong/Streamlining-CRISPR-Screen-Analysis/Streamlining-CRISPR-Screen-Analysis/data/demo_benchmark.h5ad'
    obs: 'perturbation', 'celltype'
    var: 'gene_symbols'
    uns: 'description'
First obs rows:
         perturbation celltype
cell                          
cell_000    perturb_3  NK cell
cell_001         ctrl   T cell
cell_002         ctrl   B cell
cell_003         ctrl  NK cell
cell_004    perturb_4  NK cell
First var rows:
         gene_symbols
gene                 
gene_000         G000
gene_001         G001
gene_002         G002
gene_003         G003
gene_004         G004


AnnData(path=/Users/dujinhong/Library/CloudStorage/OneDrive-TheUniversityOfHongKong/Streamlining-CRISPR-Screen-Analysis/Streamlining-CRISPR-Screen-Analysis/data/demo_benchmark.h5ad, mode='r')

In [3]:
adata_ro.obs.head()

,perturbation,celltype
cell,,
cell_000,perturb_3,NK cell
cell_001,ctrl,T cell
cell_002,ctrl,B cell
cell_003,ctrl,NK cell
cell_004,perturb_4,NK cell


In [4]:
adata_ro.var.head()

,gene_symbols
gene,
gene_000,G000
gene_001,G001
gene_002,G002
gene_003,G003
gene_004,G004


## Quality control with ``scr.pp``

The preprocessing namespace mirrors Scanpy's API. Each function returns a new ``scr.AnnData`` object backed by an ``.h5ad`` file.

In [5]:
qc_result = scr.quality_control_summary(
    adata_ro,
    perturbation_column='perturbation',
    min_genes=5,
    min_cells_per_perturbation=5,
    output_dir=OUTPUT_DIR,
    data_name='tutorial_filtered',
)
qc_preview = pd.DataFrame(
    {
        'perturbation': adata_ro.obs['perturbation'],
        'qc_pass': qc_result.cell_mask,
    }
)
adata_ro = qc_result.filtered
qc_preview.head()


,perturbation,qc_pass
cell,,
cell_000,perturb_3,True
cell_001,ctrl,True
cell_002,ctrl,True
cell_003,ctrl,True
cell_004,perturb_4,True


## Pseudobulk aggregation with ``scr.pb``

Aggregate cells per perturbation on disk. The returned object can be inspected lazily, similar to the original dataset.

In [6]:
adata_pb_ro = scr.pb.average_log_expression(
    adata_ro,
    perturbation_column='perturbation',
    output_dir=OUTPUT_DIR,
    data_name='tutorial_avg_log_expr',
)
adata_pb_ro.var.head()

""
gene
gene_000
gene_001
gene_002
gene_003
gene_004


## Differential expression with ``scr.tl``

Run a Wilcoxon test that mirrors ``scanpy.tl.rank_genes_groups``. ``scr.AnnData.uns`` entries also provide ``preview`` and ``load`` helpers.

In [7]:
adata_ro = scr.tl.rank_genes_groups(
    adata_ro,
    perturbation_column='perturbation',
    method='wilcoxon',
    output_dir=OUTPUT_DIR,
    data_name='tutorial_wilcoxon',
)
adata_ro.uns['rank_genes_groups'].preview()

{'auc': array([(0.4994709 , 0.49968774, 0.49782313, 0.49876543, 0.49914966),
        (0.49647266, 0.4950039 , 0.49959184, 0.49823633, 0.49710884),
        (0.49929453, 0.49875098, 0.4970068 , 0.49700176, 0.49761905),
        (0.49726631, 0.49968774, 0.49659864, 0.49559083, 0.49438776),
        (0.49497354, 0.49703357, 0.49591837, 0.49541446, 0.49285714)],
       dtype=[('perturb_3', '<f8'), ('perturb_4', '<f8'), ('perturb_5', '<f8'), ('perturb_2', '<f8'), ('perturb_1', '<f8')]),
 'logfoldchanges': array([(3.62907616, 3.64885047, 3.87677801, 3.50225363, 3.30489009),
        (3.63827803, 4.17053469, 3.19142224, 3.55497232, 3.48143852),
        (3.1631486 , 3.34009363, 3.46336139, 3.5271415 , 3.28300479),
        (3.10453219, 2.93297383, 3.06021941, 3.5753096 , 3.32185082),
        (3.37271662, 3.5236534 , 3.14337469, 2.81544313, 3.08004336)],
       dtype=[('perturb_3', '<f8'), ('perturb_4', '<f8'), ('perturb_5', '<f8'), ('perturb_2', '<f8'), ('perturb_1', '<f8')]),
 'names': array([('ge

Use ``load`` to materialise the complete differential expression tables for downstream analysis.

In [8]:
wilcoxon = adata_ro.uns['rank_genes_groups'].load()
wilcoxon_full = adata_ro.uns['rank_genes_groups_full'].load()
groups = list(wilcoxon['names'].dtype.names)
group = groups[0]
group_index = groups.index(group)
top_genes = wilcoxon['names'][group]
pd.DataFrame({
    'gene': top_genes,
    'logfoldchange': wilcoxon_full['logfoldchanges'][group_index][: top_genes.size],
    'pval_adj': wilcoxon_full['pvals_adj'][group_index][: top_genes.size],
}).head()


,gene,logfoldchange,pval_adj
0,gene_024,-0.276430,0.040696
1,gene_025,0.170924,0.258447
2,gene_022,-0.203878,0.137966
3,gene_023,-0.094676,0.138738
4,gene_021,-0.695081,0.027262


The ``scr.AnnData`` handles close themselves when their Python objects go out of scope, so no explicit cleanup is required.